# Projet 10 – Prédire l'attrition des employés (ESN)
## Étape 3 : Premiers modèles de classification

**Objectif :** entraîner quatre modèles de complexité croissante (Dummy, linéaire, deux modèles d'arbres), les évaluer sur le jeu d'apprentissage et sur le jeu de test, et mesurer l'overfitting.

**Plan du notebook**
1. Chargement des données et séparation train / test
2. Modèle Dummy (référence)
3. Modèle linéaire : régression logistique (sans puis avec mise à l'échelle)
4. Modèle non linéaire : Random Forest
5. Modèle de boosting : XGBoost
6. Comparaison et conclusions

**Métrique clé :** le **recall de la classe 1** (part des vrais départs retrouvés), car l'objectif est de repérer les employés qui risquent de partir. L'accuracy est trompeuse ici : 84 % des employés sont restés.

---
## 1. Chargement et séparation train / test
`X` et `y` sont obtenus avec les fonctions de `preparation.py` (étape 2).

In [1]:
from preparation import charger_donnees, preparer_donnees

X, y = preparer_donnees(charger_donnees())
print(X.shape, y.shape)

(1470, 37) (1470,)


### Séparation train / test
- 80 % pour l'apprentissage, 20 % pour le test.
- `stratify=y` : les deux jeux gardent la même proportion de départs (≈ 16 %), indispensable avec une cible déséquilibrée.

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(1176, 37) (294, 37)
0.16156462585034015 0.1598639455782313


**Contrôle :** 1 176 employés en train et 294 en test, avec ≈ 16 % de partis dans chaque jeu : la stratification a bien fonctionné.

---
## 2. Modèle Dummy (référence)
Il prédit toujours la classe majoritaire (« resté »). C'est la barre à dépasser : un vrai modèle doit faire mieux, notamment sur le recall de la classe 1.

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

print("=== TRAIN ===")
print(classification_report(y_train, dummy.predict(X_train), zero_division=0))
print("=== TEST ===")
print(classification_report(y_test, dummy.predict(X_test), zero_division=0))

=== TRAIN ===
              precision    recall  f1-score   support

           0       0.84      1.00      0.91       986
           1       0.00      0.00      0.00       190

    accuracy                           0.84      1176
   macro avg       0.42      0.50      0.46      1176
weighted avg       0.70      0.84      0.76      1176

=== TEST ===
              precision    recall  f1-score   support

           0       0.84      1.00      0.91       247
           1       0.00      0.00      0.00        47

    accuracy                           0.84       294
   macro avg       0.42      0.50      0.46       294
weighted avg       0.71      0.84      0.77       294



**Conclusion :** accuracy de 0,84 mais **recall de la classe 1 = 0** : le Dummy ne détecte aucun départ. Cette accuracy ne vient que du déséquilibre des classes, elle n'a aucune valeur. Train et test sont identiques (le Dummy n'apprend rien, donc pas d'overfitting).

---
## 3. Modèle linéaire : régression logistique
### 3.1 Sans mise à l'échelle

In [4]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)

print("=== TRAIN ===")
print(classification_report(y_train, logreg.predict(X_train), zero_division=0))
print("=== TEST ===")
print(classification_report(y_test, logreg.predict(X_test), zero_division=0))

=== TRAIN ===
              precision    recall  f1-score   support

           0       0.90      0.98      0.94       986
           1       0.83      0.43      0.56       190

    accuracy                           0.89      1176
   macro avg       0.86      0.70      0.75      1176
weighted avg       0.89      0.89      0.88      1176

=== TEST ===
              precision    recall  f1-score   support

           0       0.89      0.99      0.94       247
           1       0.86      0.38      0.53        47

    accuracy                           0.89       294
   macro avg       0.88      0.69      0.73       294
weighted avg       0.89      0.89      0.87       294



c:\Users\PC\Desktop\COURS\DEVELOPPEUR IA\P10 - MACHINE LEARNING\Projet\projet10-attrition-esn\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


**Conclusion :** premier vrai gain (test : recall 0,38, precision 0,86). Mais sklearn émet un `ConvergenceWarning` : les colonnes ont des échelles très différentes (`revenu_mensuel` en milliers, variables 0/1…), le modèle n'a pas fini de converger.

### 3.2 Mise à l'échelle des données
`StandardScaler` : moyenne 0 et écart-type 1 pour chaque colonne. Il est **ajusté sur le train uniquement** (`fit_transform`), puis appliqué au test (`transform`), afin d'éviter toute fuite d'information du test vers l'apprentissage (*data leakage*).

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 3.3 Avec mise à l'échelle

In [6]:
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_scaled, y_train)

print("=== TRAIN ===")
print(classification_report(y_train, logreg.predict(X_train_scaled), zero_division=0))
print("=== TEST ===")
print(classification_report(y_test, logreg.predict(X_test_scaled), zero_division=0))

=== TRAIN ===
              precision    recall  f1-score   support

           0       0.90      0.98      0.94       986
           1       0.81      0.46      0.59       190

    accuracy                           0.90      1176
   macro avg       0.86      0.72      0.76      1176
weighted avg       0.89      0.90      0.88      1176

=== TEST ===
              precision    recall  f1-score   support

           0       0.88      0.96      0.92       247
           1       0.62      0.34      0.44        47

    accuracy                           0.86       294
   macro avg       0.75      0.65      0.68       294
weighted avg       0.84      0.86      0.84       294



#### Matrice de confusion (régression logistique avec scaling, jeu de test)
Lecture : les **lignes** donnent la vérité, les **colonnes** la prédiction.

In [7]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, logreg.predict(X_test_scaled)))

[[237  10]
 [ 31  16]]


| | prédit « resté » | prédit « parti » |
|---|---|---|
| **vrai resté** (247) | 237 | 10 (fausses alertes) |
| **vrai parti** (47) | 31 (partis ratés) | 16 (partis retrouvés) |

- **Recall** de la classe 1 = 16 / (16 + 31) = 0,34.
- **Precision** de la classe 1 = 16 / (16 + 10) = 0,62.
- Le modèle rate près de 2 départs sur 3 (31 sur 47), mais déclenche peu de fausses alertes (10).

**Conclusion :** le warning a disparu. Test : recall 0,34 (≈ 16 des 47 partis retrouvés), precision 0,62. L'écart avec le modèle sans scaling est faible : avec seulement 47 partis dans le test, 2 employés de plus ou de moins font varier le recall de 4 points. L'écart train / test (recall 0,46 contre 0,34) montre un léger overfitting.

---
## 4. Modèle non linéaire : Random Forest
Modèle à base d'arbres, sans réglage (paramètres par défaut) et sans mise à l'échelle (inutile pour les arbres).

In [8]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

print("=== TRAIN ===")
print(classification_report(y_train, rf.predict(X_train), zero_division=0))
print("=== TEST ===")
print(classification_report(y_test, rf.predict(X_test), zero_division=0))

=== TRAIN ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       986
           1       1.00      1.00      1.00       190

    accuracy                           1.00      1176
   macro avg       1.00      1.00      1.00      1176
weighted avg       1.00      1.00      1.00      1176

=== TEST ===
              precision    recall  f1-score   support

           0       0.85      0.98      0.91       247
           1       0.56      0.11      0.18        47

    accuracy                           0.84       294
   macro avg       0.70      0.55      0.55       294
weighted avg       0.81      0.84      0.80       294



**Conclusion :** **overfitting très fort**. Le modèle est parfait en train (1,00 partout) mais s'effondre en test : recall 0,11 (≈ 5 des 47 partis retrouvés), precision 0,56. Il a appris les employés du train par cœur au lieu de généraliser. Son accuracy de 0,84 est la même que celle du Dummy.

---
## 5. Modèle de boosting : XGBoost
Autre modèle non linéaire à base d'arbres, construits l'un après l'autre : chaque arbre corrige les erreurs des précédents. Paramètres par défaut et sans mise à l'échelle (inutile pour les arbres), pour comparer dans les mêmes conditions que le Random Forest.

In [9]:
from xgboost import XGBClassifier

xgb = XGBClassifier(random_state=42, eval_metric="logloss")
xgb.fit(X_train, y_train)

print("=== TRAIN ===")
print(classification_report(y_train, xgb.predict(X_train), zero_division=0))
print("=== TEST ===")
print(classification_report(y_test, xgb.predict(X_test), zero_division=0))

=== TRAIN ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       986
           1       1.00      1.00      1.00       190

    accuracy                           1.00      1176
   macro avg       1.00      1.00      1.00      1176
weighted avg       1.00      1.00      1.00      1176

=== TEST ===
              precision    recall  f1-score   support

           0       0.88      0.96      0.92       247
           1       0.59      0.28      0.38        47

    accuracy                           0.85       294
   macro avg       0.73      0.62      0.65       294
weighted avg       0.83      0.85      0.83       294



**Conclusion :** **overfitting très fort**, comme pour le Random Forest : le modèle est parfait en train (1,00 partout) mais baisse nettement en test. Il généralise pourtant mieux : recall 0,28 (≈ 13 des 47 partis retrouvés) contre 0,11 pour le Random Forest, avec une precision de 0,59. Il reste en dessous de la régression logistique (recall 0,34). Les deux modèles d'arbres apprennent le train par cœur : il faudra les contraindre (profondeur, régularisation) à l'étape suivante.

---
## 6. Comparaison des modèles
Résultats sur la **classe 1 (partis), jeu de test** :

| Modèle | Recall | Precision | F1 | Overfitting |
|---|---|---|---|---|
| Dummy | 0,00 | 0,00 | 0,00 | non |
| Régression logistique (avec scaling) | 0,34 | 0,62 | 0,44 | léger |
| Random Forest (sans réglage) | 0,11 | 0,56 | 0,18 | très fort |
| XGBoost (sans réglage) | 0,28 | 0,59 | 0,38 | très fort |

### Synthèse
- Le jeu de données est **difficile** : aucun modèle non linéaire non réglé ne dépasse la régression logistique.
- Les deux modèles d'arbres sont parfaits en train (1,00) et bien moins bons en test : **overfitting**. XGBoost généralise mieux que le Random Forest.
- Tous les modèles **ratent la majorité des départs** : c'est l'effet du déséquilibre des classes (16 % de partis).
- L'accuracy est trompeuse ici ; on suit le **recall** (et la precision) de la classe 1.

**Prochaine étape :** améliorer les modèles (gestion du déséquilibre, profondeur des arbres, validation croisée, seuil de décision).